# Запуск приложения «Ассистент по лекциям» в Colab
Streamlit (интерфейс) + FastAPI (сервер с моделями).

1. *Среда выполнения → Сменить среду выполнения → T4 GPU*.
2. Запускайте ячейки по порядку. Последняя выдаст ссылку на приложение.

## 1. Код из GitHub

In [1]:
REPO = "https://github.com/ageraforreal/lecture-assistant.git"
!git clone -q {REPO}
%cd lecture-assistant

/content/lecture-assistant


## 2. Библиотеки

In [2]:
!pip -q install -r requirements.txt

## 3. Тесты

In [3]:
!python -m pytest -v

============================= test session starts ==============================
platform linux -- Python 3.13.16, pytest-8.4.2, pluggy-1.7.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content/lecture-assistant
plugins: platformdirs-4.13.5, langsmith-0.14.12, anyio-4.15.1, typeguard-4.6.0
collected 19 items                                                             

tests/test_api.py::test_health PASSED                                    [  5%]
tests/test_api.py::test_transcribe PASSED                                [ 10%]
tests/test_api.py::test_transcribe_rejects_not_audio PASSED              [ 15%]
tests/test_api.py::test_summary PASSED                                   [ 21%]
tests/test_api.py::test_summary_empty_text PASSED                        [ 26%]
tests/test_api.py::test_summary_missing_field PASSED                     [ 31%]
tests/test_api.py::test_ask PASSED                                       [ 36%]
tests/test_api.py::test_ask_empty_fields[payload0] PASSED 

## 4. Запуск сервера FastAPI и интерфейса Streamlit
Модели загружаются при первом запросе (1–3 минуты).

In [4]:
!nohup uvicorn api:app --port 8000 > api.log 2>&1 &
!nohup streamlit run app.py --server.port 8501 --server.headless true > ui.log 2>&1 &
!sleep 5 && curl -s localhost:8000/health

{"status":"ok"}

## 5. Публичная ссылка на приложение
Откройте ссылку `https://...trycloudflare.com`. Ячейка остаётся запущенной, пока работает приложение.

In [5]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
import re, subprocess
p = subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:8501"],
                     stderr=subprocess.PIPE, text=True)
for line in p.stderr:
    m = re.search(r"https://[\w-]+\.trycloudflare\.com", line)
    if m:
        print("Приложение:", m.group())
        break

Приложение: https://contemporary-drum-hrs-perfectly.trycloudflare.com


## Если что-то не работает
Логи сервера и интерфейса:

In [6]:
!tail -n 30 api.log ui.log

==> api.log <==
INFO:     Started server process [1180]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)
INFO:     127.0.0.1:52424 - "GET /health HTTP/1.1" 200 OK

==> ui.log <==


2026-10-29 12:37:47.496 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.87.4.175:8501

